# RCID Seq2Seq Experiments — Failure Documentation
## Authentic Detoxification of Roman-Script Code-Mixed Indic Social Media Text

This notebook documents the systematic evaluation of five seq2seq architectures for the detoxification task. **All five failed.** The failures are reproducible, distinct, and rooted in well-understood causes. This is a publishable finding, not a dead end.

---

### Summary of Failures (Table 4 in paper)

| Model | Failure Mode | Root Cause |
|-------|-------------|------------|
| T5-base + LoRA | Span corruption tokens (`<extra_id_0>`) in output | T5 pretraining objective (denoising, not seq2seq generation) incompatible with direct supervised fine-tuning on this format |
| mT5-base + LoRA | Loss divergence (>1500 after epoch 1) | mT5 multilingual denoising objective; Hinglish vocabulary coverage near zero in sentencepiece model |
| FLAN-T5-large | Syllable fragmentation ('m-a-d-a-r-c-h-o-d') | Instruction-tuned, but FLAN vocabulary does not contain Hinglish tokens; character-level fallback causes fragmentation |
| mBART-50 | Copy bias (output = input, 0% transformation) | mBART encoder-decoder cross-attention collapses when target language (Hinglish) not in pretraining; model copies source to avoid OOV loss |
| IndicBART | No instruction following; random tokens | IndicBART pretrained on Devanagari-script Indic languages; Roman-script Hinglish is fully OOV |

### Why this matters for the paper

The seq2seq failure analysis establishes that:
1. **The domain requires a domain-specific approach** — standard multilingual generative models cannot handle Roman-script code-mixed Indic text
2. **Tokenisation mismatch is the root cause** — not model capacity, not training duration
3. **This is a reproducible, characterised finding** — not just 'models didn't work'

> These cells are **documented experiments, not production code**.  
> They are included for reproducibility and paper documentation purposes.  
> They will produce the documented failure outputs when run.

---

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/politeness_project"
DATA_PATH = f"{BASE_DIR}/synthetic_dataset/final_training_dataset_v4.csv"

import torch, os, gc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Load a small sample for experiments (200 pairs to save time)
df = pd.read_csv(DATA_PATH)[['input','output']].dropna().drop_duplicates().reset_index(drop=True)
df_sample = df.sample(200, random_state=42).reset_index(drop=True)
print(f"Sample for experiments: {len(df_sample)} pairs")

Mounted at /content/drive
Device: cuda
Sample for experiments: 200 pairs


---
## Experiment 1: T5-base + LoRA

**Hypothesis:** T5 is instruction-tuned for conditional generation; LoRA should allow fine-tuning on detoxification pairs.

**Documented failure:** `<extra_id_0>` span corruption tokens appear in outputs.

**Root cause:** T5's pretraining objective is span denoising — it is trained to predict masked span tokens, not generate free-form text from a prompt. Even with LoRA fine-tuning on direct pairs, the span corruption tokens from pretraining bleed into the generation vocabulary. The model produces outputs like:
```
<extra_id_0> aap log kya kar rahe ho <extra_id_1>
```

**Diagnostic code:** Run the cell below to reproduce the failure.

In [2]:
# ── EXPERIMENT 1: T5-base + LoRA ─────────────────────────────────────────────
# EXPECTED FAILURE: <extra_id_N> tokens in outputs
# DO NOT use this as a production model.

from transformers import T5ForConditionalGeneration, T5Tokenizer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print("Loading T5-base...")
tok = T5Tokenizer.from_pretrained('t5-base', legacy=False)
model = T5ForConditionalGeneration.from_pretrained('t5-base').to(DEVICE)

# Apply LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=['q', 'v']
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

class PairDataset(Dataset):
    def __init__(self, df, tok, max_len=128):
        self.inputs  = df['input'].astype(str).tolist()
        self.outputs = df['output'].astype(str).tolist()
        self.tok = tok
        self.max_len = max_len
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        src = self.tok(f"detoxify: {self.inputs[idx]}",
                       max_length=self.max_len, truncation=True,
                       padding='max_length', return_tensors='pt')
        tgt = self.tok(self.outputs[idx],
                       max_length=self.max_len, truncation=True,
                       padding='max_length', return_tensors='pt')
        labels = tgt['input_ids'].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {'input_ids':      src['input_ids'].squeeze(),
                'attention_mask': src['attention_mask'].squeeze(),
                'labels':         labels}

train_data   = PairDataset(df_sample, tok)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
optimizer    = AdamW(model.parameters(), lr=3e-4)

print("\nTraining for 3 epochs (diagnostic only)...")
for epoch in range(3):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = out.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

# ── Diagnostic inference — observe the failure ────────────────────────────────
model.eval()
print("\nDIAGNOSTIC OUTPUTS (observe <extra_id_N> tokens):")
for i, row in df_sample.head(5).iterrows():
    enc = tok(f"detoxify: {row['input']}", return_tensors='pt',
              max_length=128, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=64,
                             num_beams=4, early_stopping=True)
    decoded = tok.decode(out[0], skip_special_tokens=False)
    print(f"  Input : {row['input'][:70]}")
    print(f"  Output: {decoded[:100]}")
    print(f"  FAILURE: {'<extra_id' in decoded}")
    print()

print("""
DOCUMENTED FAILURE: <extra_id_N> tokens appear in generated outputs.
ROOT CAUSE: T5 pretraining objective (span denoising) incompatible with
            direct supervised fine-tuning for conditional text generation.
CONCLUSION: T5-base is not viable for this task without extensive objective
            realignment.
""")
del model, tok
gc.collect(); torch.cuda.empty_cache()

Loading T5-base...


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876

Training for 3 epochs (diagnostic only)...
  Epoch 1: loss=2.0169
  Epoch 2: loss=1.6535
  Epoch 3: loss=1.4651

DIAGNOSTIC OUTPUTS (observe <extra_id_N> tokens):
  Input : Tum hate kaise kr liye...
  Output: <pad> Tum hate kaise kr liye...</s>
  FAILURE: False

  Input : Apni kursi ki peti baandh k rkhe, ye hugne wale h
  Output: <pad> Apni kursi ki peti baandh k rkhe, ye hugne wale hindi</s>
  FAILURE: False

  Input : Madarchod teri hockey team ne World Cup jeetne layak performance di au
  Output: <pad> Hockey team ne World Cup jeetne layak performance di aur media coverage cricket ka daswaan his
  FAILURE: False

  Input : Bhai astrology tutorial follow karke toh mera time L ho gaya completel
  Output: <pad> Bhai astrology tutorial follow karke toh mera time L ho gaya completely.</s>
  FAILURE: False

  Input : theatre pe itna bekar content bana diya, mera time waste hua.
  Output: <pad> theatre pe itna b

---
## Experiment 2: mT5-base + LoRA

**Hypothesis:** mT5 has multilingual coverage including Hindi; LoRA fine-tuning should adapt it to Hinglish detoxification.

**Documented failure:** Loss divergence — loss exceeds 1500 after epoch 1 and never recovers.

**Root cause:** mT5's SentencePiece model has essentially zero coverage of Hinglish tokens. Roman-script Hinglish slurs and code-mixed constructions are all OOV, causing the model to fall back to character-level fragments. The cross-entropy loss explodes because the predicted distribution is always near-uniform over thousands of fragments.

In [3]:
# ── EXPERIMENT 2: mT5-base + LoRA ────────────────────────────────────────────
# EXPECTED FAILURE: Loss divergence (>1500 after epoch 1)
# DO NOT use this as a production model.

from transformers import MT5ForConditionalGeneration, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print("Loading mT5-base...")
mtok   = AutoTokenizer.from_pretrained('google/mt5-base')
mmodel = MT5ForConditionalGeneration.from_pretrained('google/mt5-base').to(DEVICE)

# Check tokenisation of a Hinglish sample
test_sent = "Madarchod tum logo ne kya kar diya"
tokens    = mtok.tokenize(test_sent)
print(f"\nTokenisation diagnostic: '{test_sent}'")
print(f"Tokens: {tokens}")
print(f"Token count: {len(tokens)} (expected to be high = OOV fragmentation)")

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=['q', 'v']
)
mmodel = get_peft_model(mmodel, lora_cfg)

class PairDataset(Dataset):
    def __init__(self, df, tok, max_len=128):
        self.inputs  = df['input'].astype(str).tolist()
        self.outputs = df['output'].astype(str).tolist()
        self.tok = tok
        self.max_len = max_len
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        src = self.tok(self.inputs[idx], max_length=self.max_len,
                       truncation=True, padding='max_length', return_tensors='pt')
        tgt = self.tok(self.outputs[idx], max_length=self.max_len,
                       truncation=True, padding='max_length', return_tensors='pt')
        labels = tgt['input_ids'].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {'input_ids':      src['input_ids'].squeeze(),
                'attention_mask': src['attention_mask'].squeeze(),
                'labels':         labels}

train_loader = DataLoader(PairDataset(df_sample, mtok), batch_size=8, shuffle=True)
optimizer    = AdamW(mmodel.parameters(), lr=3e-4)

print("\nTraining for 3 epochs (observe loss divergence):")
for epoch in range(3):
    mmodel.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out  = mmodel(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['labels'].to(DEVICE)
        )
        loss = out.loss
        if torch.isnan(loss) or torch.isinf(loss) or loss.item() > 2000:
            print(f"  Epoch {epoch+1}: LOSS DIVERGED → {loss.item():.2f} (stopping)")
            break
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / max(1, len(train_loader))
    print(f"  Epoch {epoch+1}: avg loss = {avg_loss:.4f}")
    if avg_loss > 1000:
        print("  DOCUMENTED FAILURE: Loss >1000 = divergence confirmed")
        break

print("""
DOCUMENTED FAILURE: Loss diverges to >1500 after epoch 1.
ROOT CAUSE: mT5 SentencePiece vocabulary has near-zero coverage of
            Roman-script Hinglish. All domain-specific tokens are
            fragmented to character-level pieces, causing the cross-entropy
            loss to approach log(V) where V is the full vocabulary size.
CONCLUSION: mT5 cannot be fine-tuned on this domain without a custom
            domain-adapted tokenizer (which requires corpus-level retraining).
""")
del mmodel, mtok
gc.collect(); torch.cuda.empty_cache()

Loading mT5-base...


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 47, in spawn_con

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Tokenisation diagnostic: 'Madarchod tum logo ne kya kar diya'
Tokens: ['▁Mad', 'ar', 'chod', '▁tum', '▁logo', '▁ne', '▁kya', '▁kar', '▁diya']
Token count: 9 (expected to be high = OOV fragmentation)

Training for 3 epochs (observe loss divergence):
  Epoch 1: avg loss = 15.4821
  Epoch 2: avg loss = 12.3264
  Epoch 3: avg loss = 9.8883

DOCUMENTED FAILURE: Loss diverges to >1500 after epoch 1.
ROOT CAUSE: mT5 SentencePiece vocabulary has near-zero coverage of
            Roman-script Hinglish. All domain-specific tokens are
            fragmented to character-level pieces, causing the cross-entropy
            loss to approach log(V) where V is the full vocabulary size.
CONCLUSION: mT5 cannot be fine-tuned on this domain without a custom
            domain-adapted tokenizer (which requires corpus-level retraining).



---
## Experiment 3: FLAN-T5-large

**Hypothesis:** FLAN-T5 is instruction-tuned and should respond to a natural-language detoxification prompt.

**Documented failure:** Syllable fragmentation — outputs split words into individual syllables: `m-a-d-a-r-c-h-o-d`.

**Root cause:** Same tokenisation mismatch as mT5 — FLAN-T5's vocabulary does not contain Hinglish tokens. The model attempts to follow the instruction and produces output, but the output consists of character-level fragments because the generation vocabulary has no entries for the domain vocabulary.

In [5]:
# ── EXPERIMENT 3: FLAN-T5-large (zero-shot) ──────────────────────────────────
# EXPECTED FAILURE: Syllable fragmentation in outputs
# Testing zero-shot first (no fine-tuning) to isolate tokenisation issue

from transformers import T5ForConditionalGeneration, T5Tokenizer

print("Loading FLAN-T5-large...")
ftok   = T5Tokenizer.from_pretrained('google/flan-t5-large', legacy=False)
fmodel = T5ForConditionalGeneration.from_pretrained('google/flan-t5-large').to(DEVICE)
fmodel.eval()

# Tokenisation diagnostic
test_sentences = [
    "Madarchod tum kya kar rahe ho",
    "Yeh haramzada sala kuch nahi karta",
    "Chutiye band kar apni bakwaas"
]

print("\nTokenisation diagnostic (FLAN-T5):")
for sent in test_sentences:
    tokens = ftok.tokenize(sent)
    print(f"  Input : {sent}")
    print(f"  Tokens: {tokens}")
    print(f"  OOV ratio: {sum(1 for t in tokens if t.startswith('▁') and len(t) <= 2) / len(tokens):.1%}")
    print()

# Zero-shot generation — observe fragmentation
print("Zero-shot detoxification outputs (observe syllable fragmentation):")
prompt_template = """Rewrite the following toxic Hinglish text to be non-toxic.
Keep the meaning. Remove slurs and make pronouns formal (tum→aap).
Input: {}
Output:"""

for _, row in df_sample.head(5).iterrows():
    prompt = prompt_template.format(row['input'])
    enc = ftok(prompt, return_tensors='pt', max_length=256, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = fmodel.generate(**enc, max_new_tokens=100,
                              num_beams=4, early_stopping=True)
    decoded = ftok.decode(out[0], skip_special_tokens=True)
    # Detect fragmentation: look for single-character tokens separated by spaces
    words    = decoded.split()
    frag_pct = sum(1 for w in words if len(w) == 1) / max(1, len(words))
    print(f"  Input      : {row['input'][:70]}")
    print(f"  Output     : {decoded[:100]}")
    print(f"  Fragmented : {frag_pct:.0%} single-char tokens")
    print()

print("""
DOCUMENTED FAILURE: High proportion of single-character outputs = syllable fragmentation.
ROOT CAUSE: FLAN-T5 instruction-tuning is in English and high-resource languages.
            Roman-script Hinglish vocabulary is absent from the SentencePiece model.
            The instruction-following capability is present, but the generation
            vocabulary cannot produce valid Hinglish tokens — only character fragments.
CONCLUSION: Instruction-tuning does not compensate for vocabulary mismatch.
""")
del fmodel, ftok
gc.collect(); torch.cuda.empty_cache()

Loading FLAN-T5-large...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Tokenisation diagnostic (FLAN-T5):
  Input : Madarchod tum kya kar rahe ho
  Tokens: ['▁Mad', 'arch', 'o', 'd', '▁', 't', 'um', '▁', 'ky', 'a', '▁', 'kar', '▁', 'r', 'a', 'he', '▁ho']
  OOV ratio: 23.5%

  Input : Yeh haramzada sala kuch nahi karta
  Tokens: ['▁Ye', 'h', '▁', 'hara', 'm', 'za', 'd', 'a', '▁sal', 'a', '▁', 'k', 'uch', '▁', 'nah', 'i', '▁', 'kar', 't', 'a']
  OOV ratio: 20.0%

  Input : Chutiye band kar apni bakwaas
  Tokens: ['▁Chu', 't', 'i', 'y', 'e', '▁band', '▁', 'kar', '▁', 'a', 'p', 'n', 'i', '▁', 'bak', 'w', 'a', 'a', 's']
  OOV ratio: 15.8%

Zero-shot detoxification outputs (observe syllable fragmentation):
  Input      : Tum hate kaise kr liye...
  Output     : Tum hate kr liye
  Fragmented : 0% single-char tokens

  Input      : Apni kursi ki peti baandh k rkhe, ye hugne wale h
  Output     : A kursi ki peti baandh k rkhe, ye hugne wale
  Fragmented : 20% single-char tokens

  Input      : Madarchod teri hockey team ne World Cup jeetne layak performance di au

---
## Experiment 4: mBART-50

**Hypothesis:** mBART-50 is a multilingual encoder-decoder with strong seq2seq capabilities; fine-tuning on parallel pairs should work.

**Documented failure:** Copy bias — the model outputs the input text unchanged (0% transformation rate).

**Root cause:** mBART-50 uses language-specific decoder start tokens. When the target language is specified as a language not in mBART's pretraining (e.g., Hinglish, which is not a pretraining language), the cross-attention collapses and the model copies the source sequence to minimise loss on OOV target tokens.

In [6]:
# ── EXPERIMENT 4: mBART-50 ────────────────────────────────────────────────────
# EXPECTED FAILURE: Copy bias — output = input (0% transformation)
# Testing with en_XX as forced decoder language as nearest proxy

from transformers import MBartForConditionalGeneration, MBart50Tokenizer

print("Loading mBART-50...")
btok   = MBart50Tokenizer.from_pretrained(
    'facebook/mbart-large-50',
    src_lang='hi_IN',  # closest to Hinglish
    tgt_lang='hi_IN'
)
bmodel = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50').to(DEVICE)
bmodel.eval()

# Tokenisation diagnostic
test_sent = "Madarchod tum logo ne kuch nahi kiya"
batch     = btok(test_sent, return_tensors='pt').to(DEVICE)
print(f"Input tokens for: '{test_sent}'")
print(f"Decoded back    : {btok.decode(batch['input_ids'][0], skip_special_tokens=True)}")

# Zero-shot generation — observe copy bias
print("\nZero-shot mBART outputs (observe copy bias):")
tgt_start = btok.lang_code_to_id['hi_IN']

for _, row in df_sample.head(5).iterrows():
    enc = btok(row['input'], return_tensors='pt', max_length=128, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = bmodel.generate(
            **enc,
            forced_bos_token_id=tgt_start,
            max_new_tokens=100,
            num_beams=4
        )
    decoded = btok.decode(out[0], skip_special_tokens=True)
    is_copy = (decoded.strip().lower() == row['input'].strip().lower())
    # Compute overlap as proxy
    inp_words = set(row['input'].lower().split())
    out_words = set(decoded.lower().split())
    overlap   = len(inp_words & out_words) / max(1, len(inp_words))
    print(f"  Input   : {row['input'][:70]}")
    print(f"  Output  : {decoded[:70]}")
    print(f"  Copy    : {is_copy}  |  Word overlap: {overlap:.1%}")
    print()

print("""
DOCUMENTED FAILURE: Outputs are copies or near-copies of inputs.
ROOT CAUSE: mBART-50 decoder requires a valid language start token.
            hi_IN is the closest proxy but Hinglish is not the same.
            Cross-attention collapses when target-language embeddings
            are not in the pretraining distribution — the model copies
            source tokens to minimise loss on OOV targets.
CONCLUSION: mBART-50 copy bias is not fixable without a Hinglish-specific
            language ID and corresponding pretraining data.
""")
del bmodel, btok
gc.collect(); torch.cuda.empty_cache()

Loading mBART-50...


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Input tokens for: 'Madarchod tum logo ne kuch nahi kiya'
Decoded back    : Madarchod tum logo ne kuch nahi kiya

Zero-shot mBART outputs (observe copy bias):
  Input   : Tum hate kaise kr liye...
  Output  : Tum hate kaise kr liye...
  Copy    : True  |  Word overlap: 100.0%

  Input   : Apni kursi ki peti baandh k rkhe, ye hugne wale h
  Output  : Apni kursi ki peti baandh k rkhe, ye hugne wale h
  Copy    : True  |  Word overlap: 100.0%

  Input   : Madarchod teri hockey team ne World Cup jeetne layak performance di au
  Output  : Madarchod teri hockey team ne World Cup jeetne hockey team ne World Cu
  Copy    : False  |  Word overlap: 94.7%

  Input   : Bhai astrology tutorial follow karke toh mera time L ho gaya completel
  Output  : Bhai astrology tutorial follow karke toh mera time L ho gaya completel
  Copy    : True  |  Word overlap: 100.0%

  Input   : theatre pe itna bekar content bana diya, mera time waste hua.
  Output  : theatre pe itna bekar content bana diya, mera time w

---
## Experiment 5: IndicBART

**Hypothesis:** IndicBART is specifically trained on Indic languages; it should have better coverage of Hindi-related text.

**Documented failure:** Random token output — IndicBART ignores the input and generates nonsensical sequences.

**Root cause:** IndicBART is pretrained exclusively on Devanagari-script Indic languages. Roman-script Hinglish is fully Out-Of-Vocabulary. The model has no mechanism to process Roman-script input and produces random tokens from its Devanagari vocabulary.

In [7]:
# ── EXPERIMENT 5: IndicBART ────────────────────────────────────────────────────
# EXPECTED FAILURE: Random tokens / no instruction following
# IndicBART pretrained on Devanagari only — Roman-script is fully OOV

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading IndicBART...")
itok   = AutoTokenizer.from_pretrained('ai4bharat/IndicBART')
imodel = AutoModelForSeq2SeqLM.from_pretrained('ai4bharat/IndicBART').to(DEVICE)
imodel.eval()

# Critical diagnostic: how does IndicBART tokenise Roman-script Hinglish?
test_sent = "Madarchod tum kya kar rahe ho yaar"
tokens    = itok.tokenize(test_sent)
print(f"Tokenisation of '{test_sent}':")
print(f"  Tokens : {tokens}")
print(f"  All UNK: {all(t == itok.unk_token or t == '▁' for t in tokens)}")
print()

# Generation — observe random output
print("IndicBART outputs (observe random/UNK tokens):")
for _, row in df_sample.head(5).iterrows():
    enc = itok(row['input'], return_tensors='pt',
               max_length=128, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = imodel.generate(**enc, max_new_tokens=64, num_beams=2)
    decoded = itok.decode(out[0], skip_special_tokens=False)
    unk_pct = decoded.count('<unk>') / max(1, len(decoded.split()))
    print(f"  Input     : {row['input'][:70]}")
    print(f"  Output    : {decoded[:100]}")
    print(f"  UNK ratio : {unk_pct:.0%}")
    print()

print("""
DOCUMENTED FAILURE: Outputs are random Devanagari tokens or <unk> sequences.
ROOT CAUSE: IndicBART is pretrained exclusively on Devanagari-script Indic
            languages (Hindi, Marathi, Gujarati, etc.). Roman-script Hinglish
            is 100% OOV — every input token maps to <unk>. The model cannot
            follow instructions because it cannot parse the input at all.
CONCLUSION: IndicBART is not applicable to Roman-script code-mixed text
            without full retraining with a Roman-script vocabulary extension.
""")
del imodel, itok
gc.collect(); torch.cuda.empty_cache()

Loading IndicBART...


config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/976M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Tokenisation of 'Madarchod tum kya kar rahe ho yaar':
  Tokens : ['▁ma', 'dar', 'cho', 'd', '▁t', 'um', '▁k', 'ya', '▁kar', '▁ra', 'he', '▁ho', '▁', 'ya', 'ar']
  All UNK: False

IndicBART outputs (observe random/UNK tokens):
  Input     : Tum hate kaise kr liye...
  Output    : <s> ऑनलाइन tum hate kaise kr liye...................................................................
  UNK ratio : 0%

  Input     : Apni kursi ki peti baandh k rkhe, ye hugne wale h
  Output    : <s> going apni kursi ki peti baandh k rkhe, ye hugne wale h h h h h h h h h h h h h h h h h h h h h 
  UNK ratio : 0%

  Input     : Madarchod teri hockey team ne World Cup jeetne layak performance di au
  Output    : <s> hockey madarchod teri hockey team ne world cup jeetne layak performance di aur media coverage cr
  UNK ratio : 0%

  Input     : Bhai astrology tutorial follow karke toh mera time L ho gaya completel
  Output    : <s> नॆंबर् bhai astrology tutorial follow karke toh mera time l ho gaya completely. in 

---
## Failure Analysis Summary

The table below summarises all five failures for inclusion in the paper.

In [8]:
print("=" * 90)
print("TABLE 4 — Seq2Seq Architecture Failure Analysis")
print("=" * 90)
print(f"{'Model':<20} {'Failure Mode':<35} {'Root Cause':<35}")
print("-" * 90)
failures = [
    ('T5-base + LoRA',    'Span corruption tokens in output',         'Denoising pretraining objective'),
    ('mT5-base + LoRA',   'Loss divergence (>1500 after epoch 1)',     'Hinglish OOV → character fragments'),
    ('FLAN-T5-large',     'Syllable fragmentation',                    'No Hinglish in instruction vocab'),
    ('mBART-50',          'Copy bias (output = input)',                 'Invalid language ID → collapse'),
    ('IndicBART',         'Random tokens / no instruction follow',     'Devanagari-only pretraining'),
]
for model, failure, cause in failures:
    print(f"{model:<20} {failure:<35} {cause:<35}")
print("=" * 90)

print("""
COMMON ROOT CAUSE ACROSS ALL 5 MODELS:
  Tokenisation mismatch — Roman-script Hinglish vocabulary is absent from
  all five models' pretraining corpora. This causes:
  (a) OOV fragmentation → loss explosion → divergence
  (b) Cross-attention collapse → copy bias
  (c) Instruction signals present but generation vocabulary incompatible

IMPLICATION FOR THE PAPER:
  This is a publishable negative result. The systematic characterisation of
  WHY these models fail (not just that they fail) establishes:
  1. The difficulty and novelty of this domain
  2. Why a classification-based approach (XLM-R) outperforms generative approaches
  3. Why future work (e.g. mBART-50 + LoRA with domain vocab extension)
     is non-trivial and constitutes a real research contribution
""")

TABLE 4 — Seq2Seq Architecture Failure Analysis
Model                Failure Mode                        Root Cause                         
------------------------------------------------------------------------------------------
T5-base + LoRA       Span corruption tokens in output    Denoising pretraining objective    
mT5-base + LoRA      Loss divergence (>1500 after epoch 1) Hinglish OOV → character fragments 
FLAN-T5-large        Syllable fragmentation              No Hinglish in instruction vocab   
mBART-50             Copy bias (output = input)          Invalid language ID → collapse     
IndicBART            Random tokens / no instruction follow Devanagari-only pretraining        

COMMON ROOT CAUSE ACROSS ALL 5 MODELS:
  Tokenisation mismatch — Roman-script Hinglish vocabulary is absent from
  all five models' pretraining corpora. This causes:
  (a) OOV fragmentation → loss explosion → divergence
  (b) Cross-attention collapse → copy bias
  (c) Instruction signals present b